# Synthesize results to XLSX

Walks `/storage/results/<run_id>/`, reads each run's `manifest.json` and `metrics.json`, flattens nested keys with dot-notation, and writes one row per run to a single `.xlsx` workbook.

Rows = run_ids. Columns = the union of all flattened keys across runs.

## 1. Config

In [ ]:
from datetime import datetime
from pathlib import Path

results_dir = Path("/storage/results")
output_xlsx = results_dir / f"results_{datetime.now().strftime('%Y%m%d-%H%M%S')}.xlsx"

print("results_dir:", results_dir)
print("output_xlsx:", output_xlsx)

## 2. Load runs

Each run directory must contain at least one of `manifest.json` / `metrics.json`. Missing files are skipped silently; missing keys become NaN in the final table.

In [ ]:
import json

def flatten(obj, prefix=""):
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else k
            out.update(flatten(v, key))
    else:
        out[prefix] = obj
    return out

rows = {}
for run_dir in sorted(p for p in results_dir.iterdir() if p.is_dir()):
    row = {}
    for fname in ("manifest.json", "metrics.json"):
        fpath = run_dir / fname
        if not fpath.exists():
            continue
        try:
            row.update(flatten(json.loads(fpath.read_text())))
        except json.JSONDecodeError as e:
            print(f"skip {fpath}: {e}")
    if row:
        rows[run_dir.name] = row

print(f"loaded {len(rows)} runs")

## 3. Build DataFrame

Columns are ordered: `run_id` first (the index), then manifest fields, then metrics fields, in the order they were first encountered.

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(rows, orient="index")
df.index.name = "run_id"
df = df.sort_index()

# move run_id-ish identity columns to the front if present
front = [c for c in ("run_tag", "timestamp") if c in df.columns]
df = df[front + [c for c in df.columns if c not in front]]

print(df.shape)
df.head()

## 4. Write XLSX

In [ ]:
df.to_excel(output_xlsx, sheet_name="runs", index=True)
print("wrote:", output_xlsx)